# 05 — Évaluation finale
Métriques détaillées et chiffres à citer dans le rapport PFE.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # racine ml/ -> import src.*
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import json
m = json.load(open('../artifacts/metrics.json', encoding='utf-8'))
best = [x for x in m['models'] if x['model_name']==m['best_model']['name']][0]
llm = m['llm_baseline']
print('Best:', best['model_name'], '| LLM baseline')

## Matrice de confusion du meilleur modèle

In [ ]:
cm = np.array(best['confusion_matrix'])  # [[TP,FP],[FN,TN]]
plt.imshow(cm, cmap='Greens'); plt.colorbar()
for i in range(2):
    for j in range(2):
        plt.text(j,i,cm[i,j],ha='center',va='center',fontsize=14)
plt.xticks([0,1],['Pred +','Pred -']); plt.yticks([0,1],['Réel +','Réel -'])
plt.title('Matrice de confusion — %s' % best['model_name']); plt.show()

## Comparaison ML vs LLM (toutes métriques)

In [ ]:
comp = pd.DataFrame([best, llm]).set_index('model_name')
comp[['auc_roc','auc_pr','f1_score','precision','recall','brier_score','precision_at_20','lift_at_20']].T

## Courbe ROC

In [ ]:
from IPython.display import Image
Image('../artifacts/roc_curves.png')

## Analyse des erreurs
Les faux positifs proviennent surtout du bruit d'étiquette (~5% de clients imprévisibles) injecté volontairement dans le générateur.

In [ ]:
tp,fp = best['confusion_matrix'][0]; fn,tn = best['confusion_matrix'][1]
print('Faux positifs:', fp, '| Faux négatifs:', fn)
print('Précision: %.3f | Rappel: %.3f' % (best['precision'], best['recall']))

## Résumé pour le rapport PFE

In [ ]:
print('=== CHIFFRES CLÉS PFE ===')
print('Modèle retenu      :', best['model_name'])
print('AUC-ROC ML         :', best['auc_roc'])
print('AUC-ROC LLM        :', llm['auc_roc'])
print('Amélioration       :', m['best_model']['improvement_vs_llm'])
print('F1 / Précision@20  :', best['f1_score'], '/', best['precision_at_20'])
print('Lift@20            :', best['lift_at_20'])
print('Features / Train   :', m['n_features'], '/', m['n_training_samples'])